# Construcción de Dataset para Taller Autoencoder

El presente notebook contiene el código encargado de extraer los frames de los videos, y conformar el dataset de imágenes etiquetadas para trabajar en el taller.



## Preparación de ambiente
- Se cargan e importan dependencias
- Se conecta el ambiente con Drive

In [1]:
# Instalación de dependencias
try:
    from IPython.display import display
    import ipywidgets as widgets
except ImportError:
    !pip install opencv-python pillow ipywidgets numpy matplotlib --quiet

In [2]:
try:
    from sklearn.cluster import KMeans
except ImportError:
    !pip -q install scikit-learn
    from sklearn.cluster import KMeans

In [3]:
import os

try:
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler
except ImportError:
    !pip -q install scikit-learn
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler

In [4]:
# Conexión con Drive para permitir acceso al dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Carga de Videos
- Se recorre cada sucarpeta con videos y se carga cada uno individualmente
- Se muestran fotogramas aleatorios para garantizar la carga correcta.

In [5]:
from google.colab import files
import cv2, numpy as np, io
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# Change the video path to load from Google Drive
video_path = "/content/drive/MyDrive/Colab Notebooks/MCIC/BigData/U3/Taller Autoencoders/20242595003_teclado.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    # If the video cannot be opened from the specified path, raise an error
    raise Exception(f'No se pudo abrir el video desde: {video_path}. Verifique la ruta.')

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video cargado, {frame_count} frames")

Video cargado, 159 frames


Visualización de Frames del video

In [6]:
from ipywidgets import interact, IntSlider

def get_frame(idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, f = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(f, cv2.COLOR_BGR2RGB)

current_frame = [None]
roi_coords = [None]

def select_frame(frame_idx=0):
    frame = get_frame(frame_idx)
    if frame is None:
        print("Frame inválido.")
        return
    current_frame[0] = frame
    plt.figure(figsize=(8,6))
    plt.imshow(frame)
    plt.title(f"Frame {frame_idx}")
    plt.axis('off')
    plt.show()

interact(select_frame, frame_idx=IntSlider(min=0, max=frame_count-1, step=1, value=0));

interactive(children=(IntSlider(value=0, description='frame_idx', max=158), Output()), _dom_classes=('widget-i…

In [7]:
# Obtener el shape del video
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Shape del video (ancho, alto): ({width}, {height})")

Shape del video (ancho, alto): (1280, 720)


## Extracción de Frames de los videos
- Se recorre cada carpeta
- Se define la tasa de frames con la cual se van a extraer las imágenes.
- Se guardan las imágenes en subcarpetas con el mismo nombre que el de los videos.

In [8]:
import os
import cv2
import shutil

# Create the output directory
output_dir = os.path.join(os.path.dirname(video_path), os.path.splitext(os.path.basename(video_path))[0])

# Remove directory if it exists
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print(f"Directorio existente eliminado: {output_dir}")

os.makedirs(output_dir)
print(f"Directorio creado: {output_dir}")

# Re-open the video capture since it was released in the previous execution
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise Exception(f'No se pudo abrir el video desde: {video_path}. Verifique la ruta.')


# Reset video capture to the beginning
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# Extract frames
frame_interval = 5
frame_count = 0
saved_frame_count = 1 # Start count from 1

while True:
    ok, frame = cap.read()
    if not ok:
        break

    if frame_count % frame_interval == 0:
        # Save the frame
        frame_filename = os.path.join(output_dir, f"{os.path.splitext(os.path.basename(video_path))[0]}_{saved_frame_count:04d}.png")
        cv2.imwrite(frame_filename, frame)
        saved_frame_count += 1

    frame_count += 1

print(f"Extracción de frames completa. Se guardaron {saved_frame_count - 1} frames en {output_dir}") # Adjust the count for the print statement

# Release the video capture object
cap.release()

Directorio creado: /content/drive/MyDrive/Colab Notebooks/MCIC/BigData/U3/Taller Autoencoders/20242595003_teclado
Extracción de frames completa. Se guardaron 32 frames en /content/drive/MyDrive/Colab Notebooks/MCIC/BigData/U3/Taller Autoencoders/20242595003_teclado


## Carga de Imágenes extraídas
- Busca las imágenes guardadas en sub carpetas y las carga en matrices.
- Se guarda el nombre de la carpeta como la etiqueta de las imágenes cargadas.

(Puede ejecutarse independientemente de las anteriores celdas.)


In [ ]:
# Conexión con Drive para permitir acceso al dataset
from google.colab import drive
drive.mount('/content/drive')

- Carpeta de dónde se van a guardar las imágenes

In [12]:
base_dir = '/content/drive/MyDrive/Colab Notebooks/MCIC/BigData/U3/Taller Autoencoders/Imagenes'

- Del nombre de cada subcarpeta se extraen las etiquetas de las imágenes

In [13]:
import os

labels = []
for item in os.listdir(base_dir):
    item_path = os.path.join(base_dir, item)
    if os.path.isdir(item_path):
        labels.append(item)

print(labels)

['teclado', 'silla', 'nada', 'mesa', 'cpu', 'mouse', 'pantalla']


- Carga las imágenes de cada sub carpeta en listas y les asigna la etiqueta.



In [14]:
from PIL import Image

images = []
image_labels = []

for label in labels:
    dir_path = os.path.join(base_dir, label)
    for file_name in os.listdir(dir_path):
        file_path = os.path.join(dir_path, file_name)
        if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                img = Image.open(file_path)
                images.append(img)
                image_labels.append(label)
            except Exception as e:
                print(f"Error cargando imagen {file_path}: {e}")

print(f"Se cargaron {len(images)} imágenes y {len(image_labels)} etiquetas.")

Se cargaron 725 imágenes y 725 etiquetas.


## Preprocesamiento de las imágenes
- Se extraer la resolución de las imágenes cargadas.
- Se ajustan todas las imágenes a la resolución menor encontrada.
- Se convierten las imágenes en arreglos Numpy y se cambian los valores a escala de grises

Convierte las imágenes cargadas en arreglos NumPy para garantizar la compatibilidad con Tensorflow.



In [15]:
# Encuentra el mínimo ancho y alto de las imágenes
min_width = min(img.size[0] for img in images)
min_height = min(img.size[1] for img in images)
print(f"Menor resolución encontrada de las imágenes: {min_width}x{min_height}")

print(f"Redimensionando imágenes a: {min_width}x{min_height}")

# Resize images to the minimum resolution, convert to grayscale, and then to a NumPy array
images_np = np.array([np.array(img.resize((min_width, min_height)).convert('L')) for img in images]).astype('float32') / 255.0

# Convert labels to a NumPy array
image_labels_np = np.array(image_labels)

# Print the shapes of the resulting NumPy arrays
print("Forma del arreglo NumPy de las imágenes:", images_np.shape)
print("Forma del arreglo NumPy de las etiquetas:", image_labels_np.shape)

Menor resolución encontrada de las imágenes: 474x474
Redimensionando imágenes a: 474x474
Forma del arreglo NumPy de las imágenes: (725, 474, 474)
Forma del arreglo NumPy de las etiquetas: (725,)


- Se agrega un canal de dimensión al arreglo y se confirma la forma del arreglo.



In [17]:
import tensorflow as tf

images_np = images_np[..., tf.newaxis]

print("Dimensión de los arreglos NumPy luego de agregar el canal de dimension:", images_np.shape)

Dimensión de los arreglos NumPy luego de agregar el canal de dimension: (725, 474, 474, 1)


## Generar Datasets de entrenamiento y pruebas

- Se dividen aleatoriamente los datos conformando x_train, x_test, y_train, y_test
- Se usa semilla para garantizar reproducibilidad.

In [18]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    images_np, image_labels_np, test_size=0.2, random_state=42
)

print("x_train:", x_train.shape)
print("x_test:", x_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

x_train: (580, 474, 474, 1)
x_test: (145, 474, 474, 1)
y_train: (580,)
y_test: (145,)


In [ ]:
import os
import numpy as np

save_dir = '/content/drive/MyDrive/Colab Notebooks/MCIC/BigData/U3/Taller Autoencoders'
os.makedirs(save_dir, exist_ok=True)

np.save(os.path.join(save_dir, 'x_train_new.npy'), x_train_new)
np.save(os.path.join(save_dir, 'x_test_new.npy'), x_test_new)
np.save(os.path.join(save_dir, 'y_train_new.npy'), y_train_new)
np.save(os.path.join(save_dir, 'y_test_new.npy'), y_test_new)

print(f"Datasets saved to {save_dir}")

## Resumen
Se cargaron un total de 617 imágenes de las subcarpetas, y a cada imagen se le asignó el nombre de su subcarpeta correspondiente como etiqueta.

Las imágenes cargadas se convirtieron en un array de NumPy con forma (617, 474, 474, 1), con valores de píxel normalizados a un tipo de dato float32 entre 0 y 1. Las etiquetas se convirtieron a un array de NumPy con forma (617,).

El conjunto de datos se dividió en conjuntos de entrenamiento y prueba:

- Conjunto de entrenamiento: 493 imágenes y etiquetas (forma de x_train: (493, 474, 474, 1))

- Conjunto de prueba: 124 imágenes y etiquetas (forma de x_test: (124, 474, 474, 1))